In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import Counter

import import_ipynb
from notebooks import DataPreProcessing as dp

ModuleNotFoundError: No module named 'import_ipynb'

In [ ]:
# Get preprocessed data from DataPreProcessing notebook
X_train=dp.X_train
X_test=dp.X_test
y_train=dp.y_train
y_test=dp.y_test
clean_text=dp.clean_text

In [ ]:
class TextCNN(nn.Module):
    def __init__(self,vocab_size,embed_size,kernel_sizes,num_channels,dropout=0.7):
        super(TextCNN, self).__init__()
        self.embedding=nn.Embedding(vocab_size,embed_size,padding_idx=0)
        self.dropout=nn.Dropout(dropout)
        self.decoder=nn.Linear(sum(num_channels),2)
        self.pool=nn.AdaptiveMaxPool1d(1)
        self.relu=nn.ReLU()
        self.convs=nn.ModuleList([nn.Conv1d(embed_size,num_channels[i],kernel_size) for i,kernel_size in enumerate(kernel_sizes)])
        
    def forward(self, inputs):
        embeddings=self.embedding(inputs)
        embeddings=embeddings.permute(0,2,1)
        conv_outputs=[]
        for conv in self.convs:
            conv_out=conv(embeddings)
            conv_out=self.relu(conv_out)
            pooled=self.pool(conv_out)
            conv_outputs.append(pooled.squeeze(2))
        encoding=torch.cat(conv_outputs,dim=1)
        encoding=self.dropout(encoding)
        outputs=self.decoder(encoding)
        return outputs

In [ ]:
# Tokenization and vocabulary building
def tokenize(text):
    return text.split()

max_len=50

def build_vocab(X_train):
    counter=Counter()
    for text in X_train:
        counter.update(tokenize(text))
    vocab={word:i+1 for i,(word,_) in enumerate(counter.items())}
    vocab['<pad>']=0
    vocab['<unk>']=len(vocab)
    return vocab

def encode(text,vocab):
    return [vocab.get(word,vocab['<unk>']) for word in tokenize(text)]

def pad_sequence(seq):
    if len(seq)<max_len:
        return seq+[0]*(max_len-len(seq))
    return seq[:max_len]

In [ ]:
# Hyperparameters
embed_size=500
kernel_sizes=[3,4,5]
num_channels=[100,100,100]
learning_rate=0.001
num_epochs=5
batch_size=64
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ---------- SAVE / LOAD ----------
# SAVE_DIR is always <project_root>/models regardless of how the notebook is imported
SAVE_DIR = os.path.join(os.getcwd(), 'models')
SAVE_PATH = os.path.join(SAVE_DIR, 'textcnn.pth')

def save_model(model, vocab):
    os.makedirs(SAVE_DIR, exist_ok=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'vocab': vocab,
        'vocab_size': len(vocab),
        'embed_size': embed_size,
        'kernel_sizes': kernel_sizes,
        'num_channels': num_channels,
    }, SAVE_PATH)
    print(f'Model saved to {SAVE_PATH}')

def load_model():
    checkpoint=torch.load(SAVE_PATH, map_location=device, weights_only=False)
    loaded_vocab=checkpoint['vocab']
    m=TextCNN(checkpoint['vocab_size'],checkpoint['embed_size'],
              checkpoint['kernel_sizes'],checkpoint['num_channels'])
    m.load_state_dict(checkpoint['model_state_dict'])
    m=m.to(device)
    m.eval()
    return m, loaded_vocab

def has_saved_model():
    return os.path.exists(SAVE_PATH)

In [ ]:
# ---------- TRAIN ----------
def train(X_train, y_train, X_test, y_test):
    vocab=build_vocab(X_train)
    vocab_size=len(vocab)

    X_train_seq=[pad_sequence(encode(t,vocab)) for t in X_train]
    X_test_seq=[pad_sequence(encode(t,vocab)) for t in X_test]

    X_train_tensor=torch.tensor(X_train_seq,dtype=torch.long)
    y_train_tensor=torch.tensor(y_train.values,dtype=torch.long)
    X_test_tensor=torch.tensor(X_test_seq,dtype=torch.long)
    y_test_tensor=torch.tensor(y_test.values,dtype=torch.long)

    train_dataset=torch.utils.data.TensorDataset(X_train_tensor,y_train_tensor)
    test_dataset=torch.utils.data.TensorDataset(X_test_tensor,y_test_tensor)
    train_loader=torch.utils.data.DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
    test_loader=torch.utils.data.DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

    model=TextCNN(vocab_size,embed_size,kernel_sizes,num_channels)
    model=model.to(device)
    criterion=nn.CrossEntropyLoss()
    optimizer=optim.Adam(model.parameters(),lr=learning_rate)

    train_losses=[]
    test_losses=[]
    train_accuracies=[]
    test_accuracies=[]

    for epoch in range(num_epochs):
        model.train()
        total_loss=0; correct=0; total=0
        for batch_X,batch_y in train_loader:
            batch_X,batch_y=batch_X.to(device),batch_y.to(device)
            outputs=model(batch_X)
            loss=criterion(outputs,batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss+=loss.item()
            _,predicted=torch.max(outputs.data,1)
            total+=batch_y.size(0)
            correct+=(predicted==batch_y).sum().item()
        avg_loss=total_loss/len(train_loader)
        train_acc=100*correct/total
        train_losses.append(avg_loss)
        train_accuracies.append(train_acc)

        model.eval()
        total_loss=0; correct=0; total=0
        with torch.no_grad():
            for batch_X,batch_y in test_loader:
                batch_X,batch_y=batch_X.to(device),batch_y.to(device)
                outputs=model(batch_X)
                loss=criterion(outputs,batch_y)
                total_loss+=loss.item()
                _,predicted=torch.max(outputs.data,1)
                total+=batch_y.size(0)
                correct+=(predicted==batch_y).sum().item()
        avg_loss=total_loss/len(test_loader)
        test_acc=100*correct/total
        test_losses.append(avg_loss)
        test_accuracies.append(test_acc)
        print(f'Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} Train Acc: {train_acc:.2f}% Test Acc: {test_acc:.2f}%')

    # Auto-save after training
    save_model(model, vocab)
    return model, vocab, train_losses, test_losses, train_accuracies, test_accuracies

In [ ]:
# ---------- PREDICT ----------
def predict_message(message, model=None, vocab=None):
    """Predict whether a message is Spam or Ham.
    If model/vocab are not passed, loads the saved model automatically."""
    if model is None or vocab is None:
        model, vocab = load_model()
    cleaned=clean_text(message)
    encoded=encode(cleaned,vocab)
    padded=pad_sequence(encoded)
    tensor_input=torch.tensor([padded],dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        outputs=model(tensor_input)
        _,predicted=torch.max(outputs,1)
        probabilities=F.softmax(outputs,dim=1)
    label='Spam' if predicted.item()==1 else 'Ham'
    confidence=probabilities[0][predicted].item()*100
    return label,confidence